In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data/processed")
DATA_ROOT = Path("/home/victor/gw/cbc_pe/data/processed")

paths = {
    "P1": DATA_ROOT / "bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P1_bandpass_30_512.h5",
    "P1S": DATA_ROOT / "bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P1S_bandpass_30_512_std.h5",
    "P3": DATA_ROOT / "bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P3_whiten_bandpass_30_512.h5",
    "P4": DATA_ROOT / "bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P4_whiten_bandpass_30_512_std.h5",
}

for name, path in paths.items():
    print(name, path, path.exists())

P1 /home/victor/gw/cbc_pe/data/processed/bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P1_bandpass_30_512.h5 True
P1S /home/victor/gw/cbc_pe/data/processed/bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P1S_bandpass_30_512_std.h5 True
P3 /home/victor/gw/cbc_pe/data/processed/bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P3_whiten_bandpass_30_512.h5 True
P4 /home/victor/gw/cbc_pe/data/processed/bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P4_whiten_bandpass_30_512_std.h5 True


In [2]:
def inspect_h5(path):
    with h5py.File(path, "r") as f:
        out = {
            "X_shape": f["X"].shape,
            "y_shape": f["y"].shape,
            "num_samples": f.attrs["num_samples"],
            "num_written": f.attrs["num_written"],
            "status": f.attrs["dataset_status"],
            "duration": f.attrs["duration"],
            "length": f.attrs["length"],
            "sampling_frequency": f.attrs["sampling_frequency"],
            "processor": f.attrs["signal_processor_config"],
        }
    return out

for name, path in paths.items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    info = inspect_h5(path)
    for k, v in info.items():
        print(k, ":", v)

P1
X_shape : (5000, 3, 16384)
y_shape : (5000, 3)
num_samples : 5000
num_written : 5000
status : complete
duration : 4.0
length : 16384
sampling_frequency : 4096.0
processor : {"whitening_method": "none", "apply_highpass": true, "apply_lowpass": true, "apply_standardization": false, "output_mode": "crop_to_config", "whitening_low_frequency_cutoff": 30.0, "whitening_max_filter_duration": 0.5, "whitening_trunc_method": "hann", "highpass_frequency": 30.0, "lowpass_frequency": 512.0, "fir_order": 256, "fir_beta": 5.0, "remove_corrupted": true}
P1S
X_shape : (5000, 3, 16384)
y_shape : (5000, 3)
num_samples : 5000
num_written : 5000
status : complete
duration : 4.0
length : 16384
sampling_frequency : 4096.0
processor : {"whitening_method": "none", "apply_highpass": true, "apply_lowpass": true, "apply_standardization": true, "output_mode": "crop_to_config", "whitening_low_frequency_cutoff": 30.0, "whitening_max_filter_duration": 0.5, "whitening_trunc_method": "hann", "highpass_frequency": 30.

In [3]:
def load_vector(path, key):
    with h5py.File(path, "r") as f:
        return f[key][:]

def max_abs_diff(a, b):
    return float(np.max(np.abs(a - b)))

reference = "P3"

keys_to_compare = [
    "y",
    "parameters/mass_1",
    "parameters/mass_2",
    "parameters/chirp_mass",
    "parameters/total_mass",
    "parameters/chi_eff",
    "parameters/distance",
    "snr/network",
    "snr/target_network",
    "placement/segment_start_time",
    "windowing/used_window_duration",
]

rows = []

for other in ["P1", "P1S", "P4"]:
    for key in keys_to_compare:
        a = load_vector(paths[reference], key)
        b = load_vector(paths[other], key)

        rows.append({
            "reference": reference,
            "other": other,
            "key": key,
            "shape_ref": a.shape,
            "shape_other": b.shape,
            "max_abs_diff": max_abs_diff(a, b),
            "allclose": bool(np.allclose(a, b, rtol=1e-6, atol=1e-8)),
        })

comparison_df = pd.DataFrame(rows)
comparison_df

,reference,other,key,shape_ref,shape_other,max_abs_diff,allclose
0,P3,P1,y,"(5000, 3)","(5000, 3)",0.0,True
1,P3,P1,parameters/mass_1,"(5000,)","(5000,)",0.0,True
2,P3,P1,parameters/mass_2,"(5000,)","(5000,)",0.0,True
3,P3,P1,parameters/chirp_mass,"(5000,)","(5000,)",0.0,True
4,P3,P1,parameters/total_mass,"(5000,)","(5000,)",0.0,True
5,P3,P1,parameters/chi_eff,"(5000,)","(5000,)",0.0,True
6,P3,P1,parameters/distance,"(5000,)","(5000,)",0.0,True
7,P3,P1,snr/network,"(5000,)","(5000,)",0.0,True
8,P3,P1,snr/target_network,"(5000,)","(5000,)",0.0,True
9,P3,P1,placement/segment_start_time,"(5000,)","(5000,)",0.0,True


In [4]:
N = inspect_h5(paths["P3"])["X_shape"][0]

rng = np.random.default_rng(2026)
indices = rng.permutation(N)

n_train = int(0.70 * N)
n_val = int(0.15 * N)
n_test = N - n_train - n_val

idx_train = indices[:n_train]
idx_val = indices[n_train:n_train+n_val]
idx_test = indices[n_train+n_val:]

splits_path = DATA_ROOT / "bbh_4s_n5000_processing_benchmark_splits_train3500_val750_test750_seed2026.npz"

np.savez(
    splits_path,
    train_idx=idx_train,
    val_idx=idx_val,
    test_idx=idx_test,
)

splits_path

PosixPath('/home/victor/gw/cbc_pe/data/processed/bbh_4s_n5000_processing_benchmark_splits_train3500_val750_test750_seed2026.npz')

In [5]:
def load_X_sample(path, n=128):
    with h5py.File(path, "r") as f:
        return f["X"][:n]

rows = []

for other in ["P1", "P1S","P4"]:
    X_ref = load_X_sample(paths["P3"])
    X_other = load_X_sample(paths[other])

    rows.append({
        "reference": "P3",
        "other": other,
        "X_shape": X_ref.shape,
        "max_abs_diff": float(np.max(np.abs(X_ref - X_other))),
        "mean_abs_diff": float(np.mean(np.abs(X_ref - X_other))),
        "allclose": bool(np.allclose(X_ref, X_other)),
    })

pd.DataFrame(rows)

,reference,other,X_shape,max_abs_diff,mean_abs_diff,allclose
0,P3,P1,"(128, 3, 16384)",161.828293,17.570433,False
1,P3,P1S,"(128, 3, 16384)",154.941071,16.790873,False
2,P3,P4,"(128, 3, 16384)",154.780396,16.774183,False


In [6]:
def summarize_X(path, n=512):
    with h5py.File(path, "r") as f:
        X = f["X"][:n]

    return {
        "mean": float(X.mean()),
        "std": float(X.std()),
        "min": float(X.min()),
        "max": float(X.max()),
        "max_abs": float(np.max(np.abs(X))),
        "rms": float(np.sqrt(np.mean(X.astype(np.float64)**2))),
        "finite": bool(np.all(np.isfinite(X))),
    }

x_summary_df = pd.DataFrame([
    {"preset": name, **summarize_X(path)}
    for name, path in paths.items()
])

x_summary_df

,preset,mean,std,min,max,max_abs,rms,finite
0,P1,-1.297854e-28,9.169400e-23,-7.983945e-22,7.580751e-22,7.983945e-22,9.088018e-23,True
1,P1S,4.893613e-06,1.002494e+00,-7.613372e+00,7.787965e+00,7.787965e+00,1.002494e+00,True
2,P3,-1.026648e-04,2.207014e+01,-1.718800e+02,1.830808e+02,1.830808e+02,2.207015e+01,True
3,P4,-1.246237e-06,1.000210e+00,-7.683239e+00,7.919587e+00,7.919587e+00,1.000210e+00,True


In [7]:
splits = np.load(splits_path)

idx_train = splits["train_idx"]
idx_val = splits["val_idx"]
idx_test = splits["test_idx"]

print(len(idx_train), len(idx_val), len(idx_test))
print("overlap train/val:", len(set(idx_train) & set(idx_val)))
print("overlap train/test:", len(set(idx_train) & set(idx_test)))
print("overlap val/test:", len(set(idx_val) & set(idx_test)))
print("total unique:", len(set(idx_train) | set(idx_val) | set(idx_test)))

3500 750 750
overlap train/val: 0
overlap train/test: 0
overlap val/test: 0
total unique: 5000


## 3. CNN benchmark setup

In [10]:
import h5py
import numpy as np
from pathlib import Path

DATA_ROOT = Path("/home/victor/gw/cbc_pe/data/processed")

split_path = DATA_ROOT / "bbh_4s_n5000_processing_benchmark_splits_train3500_val750_test750_seed2026.npz"
splits = np.load(split_path)

train_idx = splits["train_idx"]

reference_path = DATA_ROOT / "bbh_processed_4s_seobnrv4opt_snr10-25_n5000_P3_whiten_bandpass_30_512.h5"

with h5py.File(reference_path, "r") as f:
    y_all = f["y"][:]          # carga todo y, pequeño: (5000, 3)
    y_train = y_all[train_idx]

y_mean = y_train.mean(axis=0).astype(np.float32)
y_std = y_train.std(axis=0).astype(np.float32)

label_names = np.array(["chirp_mass", "total_mass", "chi_eff"])

label_stats_path = DATA_ROOT / "bbh_4s_n5000_processing_benchmark_label_stats_train_only_seed2026.npz"

np.savez(
    label_stats_path,
    y_mean=y_mean,
    y_std=y_std,
    label_names=label_names,
)

print("y_mean:", y_mean)
print("y_std:", y_std)
print("saved:", label_stats_path)

y_mean: [3.7872143e+01 9.6091713e+01 3.3206679e-03]
y_std: [16.377375   34.216957    0.43591684]
saved: /home/victor/gw/cbc_pe/data/processed/bbh_4s_n5000_processing_benchmark_label_stats_train_only_seed2026.npz
